In [1]:
import pandas as pd 
import numpy as np

import sys
import os
EDHREC_REPO_PATH = '..'
sys.path.append(os.path.abspath(EDHREC_REPO_PATH))
# Set the environment variables
os.environ['APP_ENVIRONMENT'] = 'dev'
os.environ['APP_NAME'] = 'api'
os.environ['AWS_DEFAULT_REGION'] = 'us-east-2'
os.environ['APP_CONFIG_DIR'] = os.path.abspath(f'{EDHREC_REPO_PATH}/edhrec/configs/')

from edhrec.plus.query import athena
from edhrec.plus.jobs.count_warehouse.config import (
    ATHENA_WORKGROUP,
    ICEBERG_WAREHOUSE,
)

import plotly.express as px

import logging

# Add this at the top of your notebook/script
logging.getLogger('PIL').setLevel(logging.WARNING)

### Querying from Iceberg to get "depth" data

In [2]:
engine = athena.AthenaQueryEngine(athena_workgroup = ATHENA_WORKGROUP, iceberg_warehouse = ICEBERG_WAREHOUSE)


2026-04-28 12:10:03,554 INFO [edhrec.plus.query.athena] Initializing with workgroup 'edhrec-data-lake-dev' and Iceberg warehouse 's3://edhrec-decks-dev/warehouse/'


In [13]:
# starting_columns = ['commander', 'commander2', 'user_id', 'update_time']
# ending_columns = ['commander', 'commander2', 'user_id', 'first_create_time', 'first_update_date', 'last_update_date', 'total_active_time', 'num_updates']

# query_str = f"""

# WITH date_span AS (
#     SELECT MIN(update_time) as min_update_time, MAX(update_time) as max_update_time
#     FROM prod.all_decks
#     WHERE user_id IS NOT NULL and create_time IS NOT NULL
# ),

# deck_history AS (
#     SELECT commander, commander2, user_id, create_time, DATE(update_time) as update_date
#     FROM prod.all_decks
#     WHERE user_id IS NOT NULL and update_time >= create_time + INTERVAL '60' DAY
# ),

# -- limit to only one update per commander/user/day to avoid overcounting updates for users who make multiple updates in a day
# deck_history_daily_updates AS (
#     SELECT commander, commander2, user_id, update_date, MIN(create_time) as create_time, 1 as num_updates
#     FROM deck_history
#     GROUP BY commander, commander2, user_id, update_date
# ),

# -- get total number of updates by user and commander
# user_commander_update_counts AS (
#     SELECT commander, commander2, user_id, 
#         MIN(create_time) as first_create_time,
#         MIN(update_date) as first_update_date,
#         MAX(update_date) as last_update_date,
#         DATE_DIFF('day', MIN(create_time), MAX(update_date)) as total_active_time,
#         SUM(num_updates) as num_updates
#     FROM deck_history_daily_updates
#     GROUP BY commander, commander2, user_id
# )

# SELECT {', '.join(ending_columns)}
# FROM user_commander_update_counts
# ORDER BY commander, commander2, user_id
# """

# raw_results = engine.execute(query_str)

# df = pd.DataFrame(raw_results, columns=ending_columns)
df = pd.read_parquet("../edhrec-notebooks/wotc_presentation_analyses/user_deck_update_counts_after_60_day_burn_in_period.parquet", engine='pyarrow')


# ## edits to df
commanders_unquote = ['"Jenova, Ancient Calamity"','"Shantotto"', '"Slippery Eel"']
df.loc[df['commander'].isin(commanders_unquote), 'commander'] = df.loc[df['commander'].isin(commanders_unquote), 'commander'].str.replace('"', '')
commanders_unquote = ['Jenova, Ancient Calamity','Shantotto', 'Slippery Eel']
df['commanders'] = df.apply(lambda row: f"{row['commander']} + {row['commander2']}" if pd.notna(row['commander2']) else row['commander'], axis=1)

# df.to_parquet("user_deck_update_counts_after_60_day_burn_in_period.parquet", index=False)

# df


In [14]:
keep_commanders = df.groupby('commanders')['user_id'].nunique().reset_index()\
    .rename(columns={"user_id": "num_users"})\
    .query("num_users > 200")\
    ['commanders'].values.tolist()

keep_commanders

['Aang, at the Crossroads',
 'Aatchik, Emerald Radian',
 'Abaddon the Despoiler',
 "Abdel Adrian, Gorion's Ward + Candlekeep Sage",
 "Abdel Adrian, Gorion's Ward + Far Traveler",
 'Abigale, Eloquent First-Year',
 'Abomination of Llanowar',
 'Absolute Virtue',
 'Abuelo, Ancestral Echo',
 'Acererak the Archlich',
 'Aclazotz, Deepest Betrayal',
 'Adeline, Resplendent Cathar',
 'Adeliz, the Cinder Wind',
 'Admiral Beckett Brass',
 'Admiral Brass, Unsinkable',
 'Adrix and Nev, Twincasters',
 'Aegar, the Freezing Flame',
 'Aerith Gainsborough',
 'Aerith, Last Ancient',
 'Aesi, Tyrant of Gyre Strait',
 'Aeve, Progenitor Ooze',
 'Agatha of the Vile Cauldron',
 'Agent Frank Horrigan',
 'Agrus Kos, Eternal Soldier',
 'Aisha of Sparks and Smoke',
 'Ajani, Nacatl Pariah',
 'Akiri, Fearless Voyager',
 'Akiri, Line-Slinger + Ardenn, Intrepid Archaeologist',
 'Akiri, Line-Slinger + Silas Renn, Seeker Adept',
 'Akiri, Line-Slinger + Thrasios, Triton Hero',
 'Akroma, Vision of Ixidor + Rograkh, Son of 

In [15]:
df_select = df.query("commanders in @keep_commanders")

df_summary = df_select.groupby(["commanders", "commander", "commander2"], dropna = False).agg({
    'num_updates': ['mean'],
    'user_id': 'count'
}).reset_index()

df_summary.columns = ['commanders', 'commander', 'commander2', 'average_num_updates', 'num_users']
df_summary

,commanders,commander,commander2,average_num_updates,num_users
0,"Aang, at the Crossroads","Aang, at the Crossroads",NaN,2.875332,754
1,"Aatchik, Emerald Radian","Aatchik, Emerald Radian",NaN,2.877593,482
2,Abaddon the Despoiler,Abaddon the Despoiler,NaN,2.338828,2184
3,"Abdel Adrian, Gorion's Ward + Candlekeep Sage","Abdel Adrian, Gorion's Ward",Candlekeep Sage,2.808087,1558
4,"Abdel Adrian, Gorion's Ward + Far Traveler","Abdel Adrian, Gorion's Ward",Far Traveler,2.741176,425
...,...,...,...,...,...
1691,Zurgo Stormrender,Zurgo Stormrender,NaN,4.080310,5678
1692,Zurgo and Ojutai,Zurgo and Ojutai,NaN,2.754545,660
1693,"Zurgo, Thunder's Decree","Zurgo, Thunder's Decree",NaN,3.285344,1542
1694,"Zurzoth, Chaos Rider","Zurzoth, Chaos Rider",NaN,2.623881,1340


In [ ]:
commanders_to_released_at_query_str = """
WITH distinct_commander_pairs AS (
    SELECT DISTINCT commander, commander2
    FROM prod.all_decks
),
                                    
latest_release_date_card_pairs AS (
    SELECT commander, commander2, GREATEST(cr1.released_at, COALESCE(cr2.released_at, cr1.released_at)) as released_at
    FROM distinct_commander_pairs
    JOIN prod.card_ref cr1 ON distinct_commander_pairs.commander = cr1.name
    LEFT JOIN prod.card_ref cr2 ON distinct_commander_pairs.commander2 = cr2.name
)

SELECT *
FROM latest_release_date_card_pairs

"""

raw_results = engine.execute(commanders_to_released_at_query_str)

commander_name_to_released_at = pd.DataFrame(raw_results, columns=['commander', 'commander2', 'released_at'])

commander_name_to_released_at['time_since_release'] = (pd.to_datetime('today') - pd.to_datetime(commander_name_to_released_at['released_at'])).dt.days
commander_name_to_released_at['time_since_release'] = np.where(commander_name_to_released_at['time_since_release'] >= 365*2, 365*2, commander_name_to_released_at['time_since_release'])

commander_name_to_released_at.loc[commander_name_to_released_at['commander'].isin(commanders_unquote), 'commander'] = commander_name_to_released_at.loc[commander_name_to_released_at['commander'].isin(commanders_unquote), 'commander'].str.replace('"', '')


2026-04-28 12:21:33,141 DEBUG [pyathena.common] WITH distinct_commander_pairs AS (
    SELECT DISTINCT commander, commander2
    FROM prod.all_decks
),

latest_release_date_card_pairs AS (
    SELECT commander, commander2, GREATEST(cr1.released_at, COALESCE(cr2.released_at, cr1.released_at)) as released_at
    FROM distinct_commander_pairs
    JOIN prod.card_ref cr1 ON distinct_commander_pairs.commander = cr1.name
    LEFT JOIN prod.card_ref cr2 ON distinct_commander_pairs.commander2 = cr2.name
)

SELECT *
FROM latest_release_date_card_pairs


In [17]:
plot_df = df_summary.merge(
    commander_name_to_released_at,
    on=["commander", "commander2"],
    how="left"
)

plot_df.to_parquet("depth_vs_breadth_plot_df.parquet", index=False)

In [18]:
fig = px.scatter(
    plot_df,
    x="num_users",
    y="average_num_updates",
    color="time_since_release",
    color_continuous_scale=px.colors.sequential.thermal[::-1],
    hover_name="commanders",
    hover_data={
        "num_users": ":,.0f",
        "average_num_updates": ":.3f",
        "commanders": False,
    },
    labels={
        "num_users": "# users with a deck for this commander",
        "average_num_updates": "average # deck updates for this commander across users",
        "time_since_release": "days since commander release  (2 year cap)",
    },
    title="Breadth (x) vs Depth (y) for commanders in the last two years",
    log_x=True,
    width=1200,
    height=800,
    render_mode="svg",
)

fig.write_html("breadth_vs_depth_updates_after_60_day_burn_in.html")
fig.show()